# pytorch基本操作

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

In [ ]:
import torch

## 切片

In [11]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
X[0:1, 2] = 9
X

tensor([[ 0.,  1.,  9.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

## 节省内存

In [22]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)
Y = torch.zeros_like(X)

**多一份内存**

In [23]:
before = id(Y)
Y = X + Y
before == id(Y)

False

**复用内存**

In [24]:
before = id(Y)
Y[:] = X + Y
before == id(Y)

True

## 张量转换为其他Python对象

In [25]:
X = torch.arange(12, dtype=torch.float32);
A = X.numpy()
B = torch.tensor(A)
type(A), type(B)

(numpy.ndarray, torch.Tensor)

**大小为1的张量转为标量**

In [27]:
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

## 数据预处理

In [2]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Proce\n') # 列名
    f.write('NA,Pave,127500\n') # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [3]:
import pandas as pd
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Proce
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


### 处理缺失值

In [4]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = inputs.fillna(inputs.mean(numeric_only=True))
inputs

,NumRooms,Alley
0,3.0,Pave
1,2.0,NaN
2,4.0,NaN
3,3.0,NaN


In [5]:
inputs = pd.get_dummies(inputs, dummy_na=True)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0        True      False
1       2.0       False       True
2       4.0       False       True
3       3.0       False       True


In [6]:
import torch 

X, y = torch.tensor(inputs.astype(float).values), torch.tensor(outputs.values)
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500, 106000, 178100, 140000]))

## 线性代数

### Hadamard积

$A \bigodot B$ = $\begin{bmatrix}
                    a_{11}b_{11} & a_{12}b_{12} & \dots & a_{1n}b_{1n}\\
                    a_{21}b_{21} & a_{22}b_{22} & \dots & a_{2n}b_{2n}\\
                    \vdots & \vdots & \ddots & \vdots \\
                    a_{m1}b_{m1} & a_{m2}b_{m2} & \dots & a_{mn}b_{mn}
                \end{bmatrix}$

In [15]:
A = torch.arange(20, dtype=torch.float32).reshape(5, 4)
B = A.clone()
A, A * B

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [12., 13., 14., 15.],
         [16., 17., 18., 19.]]),
 tensor([[  0.,   1.,   4.,   9.],
         [ 16.,  25.,  36.,  49.],
         [ 64.,  81., 100., 121.],
         [144., 169., 196., 225.],
         [256., 289., 324., 361.]]))

### 降维

In [11]:
x = torch.arange(4, dtype=torch.float32)
x, x.sum()

(tensor([0., 1., 2., 3.]), tensor(6.))

In [12]:
A.shape, A.sum()

(torch.Size([5, 4]), tensor(190.))

In [14]:
A_sum_axis0 = A.sum(axis=0)
A_sum_axis0, A_sum_axis0.shape

(tensor([40., 45., 50., 55.]), torch.Size([4]))

In [16]:
A_sum_axis1 = A.sum(axis=1)
A_sum_axis1, A_sum_axis1.shape

(tensor([ 6., 22., 38., 54., 70.]), torch.Size([5]))

In [17]:
A.sum(axis=[0, 1]) # 结果和A.sum()相同

tensor(190.)

**平均值**

In [22]:
A.mean(), A.sum() / A.numel()

(tensor(9.5000), tensor(9.5000))

In [23]:
A.mean(axis=0), A.sum(axis=0) / A.shape[0]

(tensor([ 8.,  9., 10., 11.]), tensor([ 8.,  9., 10., 11.]))

**非降维求和**

In [26]:
sum_A = A.sum(axis=1, keepdims=True)
sum_A

tensor([[ 6.],
        [22.],
        [38.],
        [54.],
        [70.]])

In [27]:
# 由于sum_A仍保持两个轴，可以通过广播将A除以sum_A
A / sum_A

tensor([[0.0000, 0.1667, 0.3333, 0.5000],
        [0.1818, 0.2273, 0.2727, 0.3182],
        [0.2105, 0.2368, 0.2632, 0.2895],
        [0.2222, 0.2407, 0.2593, 0.2778],
        [0.2286, 0.2429, 0.2571, 0.2714]])

In [28]:
# 按某个轴计算A元素的累计总和
A.cumsum(axis=0)

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  6.,  8., 10.],
        [12., 15., 18., 21.],
        [24., 28., 32., 36.],
        [40., 45., 50., 55.]])

### 点积

相同位置的按元素乘积的和

In [31]:
y = torch.ones(4, dtype=torch.float32)
x, y, torch.dot(x, y)

(tensor([0., 1., 2., 3.]), tensor([1., 1., 1., 1.]), tensor(6.))

In [32]:
torch.sum(x * y)

tensor(6.)

### 矩阵-向量积

In [35]:
A.shape, x.shape, torch.mv(A, x)

(torch.Size([5, 4]), torch.Size([4]), tensor([ 14.,  38.,  62.,  86., 110.]))

### 矩阵-矩阵乘法

In [36]:
B = torch.ones(4, 3)
torch.mm(A, B)

tensor([[ 6.,  6.,  6.],
        [22., 22., 22.],
        [38., 38., 38.],
        [54., 54., 54.],
        [70., 70., 70.]])

### 范数

**L2范数**

$\lVert x \rVert = \lVert x \rVert_2 = \sqrt{\sum_{i = 1}^n x_i^2}$ 

In [43]:
# L2范数
u = torch.tensor([3.0, -4.0])
torch.norm(u)

tensor(5.)

**L1范数**

$\lVert x \rVert_1 = \sum_{i = 1}^n \lvert x_i \rvert$

In [44]:
# L1范数
torch.abs(u).sum()

tensor(7.)

**$L_p$范数**

$\lVert x \rVert_p = (\sum_{i=1}^n \lvert x_i \rvert^p)^{1/p}$

**矩阵的Frobenius范数**

$\lVert X \rVert_F = \sqrt{\sum_{i=1}^m \sum_{j=1}^n x_{ij}^2}$

In [45]:
torch.norm(torch.ones(4, 9))

tensor(6.)

## 练习

In [47]:
x = torch.ones(2, 3, 4)
len(x)

2

In [50]:
A / A.sum(axis=1)

RuntimeError: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1

## 自动微分

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import torch

In [3]:
x = torch.arange(4.0)
x

tensor([0., 1., 2., 3.])

In [4]:
x.requires_grad_(True) # 等价于x=torch.arange(4.0, requires_gard=True)
x.grad # 默认值是None

In [5]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [6]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

In [7]:
x.grad == 4 * x

tensor([True, True, True, True])

In [8]:
# 默认情况下，PyTorch会积累梯度，需要清除之前的值
x.grad.zero_()
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1., 1.])

### 非标量变量的反向传播

In [9]:
# 对⾮标量调⽤backward需要传⼊⼀个gradient参数，该参数指定微分函数关于self的梯度。
# 本例只想求偏导数的和，所以传递⼀个1的梯度是合适的
x.grad.zero_()
y = x * x
# 等价于y.backward(torch.ones(len(x)))
y.sum().backward()
x.grad

tensor([0., 2., 4., 6.])

### 分离计算

In [ ]:
"""
    这⾥可以分离y来返回⼀个新变量u，该变量与y具有相同的值，但丢弃计算图中如何计算y的任何信息。换句
    话说，梯度不会向后流经u到x。因此，下⾯的反向传播函数计算z=u*x关于x的偏导数，同时将u作为常数处理，
    ⽽不是z=x*x*x关于x的偏导数。
"""
x.grad.zero_()
y = x * x
u = y.detach()

z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

In [12]:
# 由于记录了y的计算结果，我们可以随后在y上调⽤反向传播，得到y=x*x关于的x的导数，即2*x。
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

tensor([True, True, True, True])